# 📊 Exploratory Data Analysis & Modeling — Household Income Prediction

**Dataset:** Regression Dataset for Household Income Analysis  
**Source:** [Kaggle - Stealth Technologies](https://www.kaggle.com/datasets/stealthtechnologies/regression-dataset-for-household-income-analysis)  
**Objective:** Phân tích khám phá dữ liệu (EDA), tiền xử lý, huấn luyện và đánh giá mô hình Random Forest Regression.

### Nội dung phân tích

**Phần I — Exploratory Data Analysis**
1. Import Libraries & Load Data
2. Data Overview & Quality Check
3. Phân phối thu nhập (Income Distribution)
4. Phân tích theo trình độ học vấn (Box Plot)
5. Phân tích theo nghề nghiệp (Grouped Bar)
6. Phân tích theo khu vực (Bar + Pie)
7. Phân tích tương quan (Correlation Heatmap)
8. Phân tích tình trạng việc làm & giới tính
9. Phân tích tình trạng hôn nhân & loại nhà ở
10. Phân tích tuổi & kinh nghiệm (Scatter)
11. Phân phối phần trăm thu nhập (Area Chart)
12. Phân phối trình độ & nghề nghiệp (Doughnut)

**Phần II — Data Preprocessing & Modeling**

13. Data Preprocessing (Log Transform, Label Encoding, Feature Engineering)
14. Train-Test Split
15. Feature Scaling
16. Model Training (5 mô hình hồi quy)
17. Model Evaluation (So sánh, Feature Importance, Learning Curve)
18. Tổng kết & Khuyến nghị

---

# PHẦN I — EXPLORATORY DATA ANALYSIS

---

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Unified color palette (consistent with Word report) ──────────────────
COLORS = {
    'primary':   '#2E86AB',
    'secondary': '#A23B72',
    'accent':    '#F18F01',
    'success':   '#2ECC71',
    'danger':    '#E74C3C',
    'info':      '#3498DB',
    'dark':      '#2C3E50',
    'palette':   ['#2E86AB', '#A23B72', '#F18F01', '#2ECC71', '#E74C3C',
                  '#9B59B6', '#1ABC9C', '#F39C12', '#3498DB', '#E67E22',
                  '#16A085', '#8E44AD', '#2980B9', '#D35400', '#27AE60'],
}

# Style settings
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 14,
})

print('Libraries imported successfully!')
print(f'Color palette loaded: {len(COLORS["palette"])} colors')

In [ ]:
import pandas as pd

# Load dataset from the local CSV file
df = pd.read_csv('../data/data.csv')

print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
df.head(10)

## 2. Data Overview & Quality Check

In [ ]:
# Data types and info
print('=== Data Types ===')
print(df.dtypes)
print(f'\n=== Memory Usage ===')
print(f'Total: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

In [ ]:
# Missing values check
missing = df.isnull().sum()
print('=== Missing Values ===')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found!')
print(f'\nTotal missing: {missing.sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# Statistical summary for numerical columns
df.describe().round(2)

In [ ]:
# Identify column types
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols = df.select_dtypes(include=['number']).columns.tolist()

print(f'Categorical columns ({len(cat_cols)}):')
for col in cat_cols:
    print(f'  - {col}: {df[col].nunique()} unique values → {df[col].unique().tolist()}')

print(f'\nNumerical columns ({len(num_cols)}):')
for col in num_cols:
    print(f'  - {col}: range [{df[col].min()}, {df[col].max()}]')

## 3. Phân phối thu nhập (Income Distribution)

So sánh phân phối thu nhập gốc với phân phối sau biến đổi log — nhằm xác định chiến lược xử lý skewness cho mô hình.

In [ ]:
# ── Income Distribution (Raw) ──
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(df['Income'], bins=80, color=COLORS['primary'], alpha=0.85, edgecolor='white')
ax.set_xlabel('Annual Income (USD)')
ax.set_ylabel('Frequency')
ax.set_title('Income Distribution (Raw)', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
ax.axvline(df['Income'].mean(),   color=COLORS['danger'],  linestyle='--', linewidth=1.5,
           label=f"Mean  ${df['Income'].mean()/1e6:.2f}M")
ax.axvline(df['Income'].median(), color=COLORS['success'], linestyle='--', linewidth=1.5,
           label=f"Median ${df['Income'].median()/1000:.0f}K")
ax.legend()

plt.tight_layout()
plt.show()

print(f"Skewness: {df['Income'].skew():.4f}  (heavily right-skewed — no log transform applied)")


In [ ]:
# Percentile analysis & IQR outlier detection
percentiles_list = [5, 10, 25, 50, 75, 90, 95, 99]
print('=== Income Percentiles ===')
for p in percentiles_list:
    val = df['Income'].quantile(p / 100)
    print(f'  P{p:2d}: ${val:>12,.0f}')

# IQR outlier detection
Q1, Q3 = df['Income'].quantile(0.25), df['Income'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['Income'] < Q1 - 1.5 * IQR) | (df['Income'] > Q3 + 1.5 * IQR)]
print(f'\n=== Outlier Analysis (IQR) ===')
print(f'  Q1: ${Q1:,.0f}  |  Q3: ${Q3:,.0f}  |  IQR: ${IQR:,.0f}')
print(f'  Lower bound: ${Q1 - 1.5 * IQR:,.0f}')
print(f'  Upper bound: ${Q3 + 1.5 * IQR:,.0f}')
print(f'  Outliers: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)')

## 4. Phân tích theo trình độ học vấn (Box Plot)

Box plot thể hiện phân phối thu nhập theo từng cấp độ học vấn, cho thấy trung vị, IQR, và outliers.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
order = ['High School', "Bachelor's", "Master's", 'Doctorate']
palette_edu = [COLORS['primary'], COLORS['secondary'], COLORS['accent'], COLORS['success']]

sns.boxplot(data=df, x='Education_Level', y='Income', order=order, palette=palette_edu, ax=ax,
            flierprops=dict(marker='o', markersize=2, alpha=0.3))
ax.set_title('Income Distribution by Education Level', fontweight='bold')
ax.set_xlabel('Education Level')
ax.set_ylabel('Annual Income (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

plt.tight_layout()
plt.show()

# Summary table
print('=== Income by Education Level ===')
edu_stats = df.groupby('Education_Level')['Income'].agg(['count', 'mean', 'median', 'std'])
for level in order:
    if level in edu_stats.index:
        row = edu_stats.loc[level]
        print(f'  {level:15s}  Count: {int(row["count"]):>5,}  '
              f'Mean: ${row["mean"]:>12,.0f}  Median: ${row["median"]:>10,.0f}  '
              f'Std: ${row["std"]:>12,.0f}')

## 5. Phân tích theo nghề nghiệp (Grouped Bar)

So sánh Mean vs Median thu nhập theo nhóm nghề nghiệp — chênh lệch lớn phản ánh phân phối lệch phải.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
occ_stats = df.groupby('Occupation')['Income'].agg(['mean', 'median']).sort_values('median', ascending=False)
x = np.arange(len(occ_stats))
width = 0.35

bars1 = ax.bar(x - width/2, occ_stats['median'], width, label='Median', color=COLORS['primary'], alpha=0.9)
bars2 = ax.bar(x + width/2, occ_stats['mean'], width, label='Mean', color=COLORS['accent'], alpha=0.9)

ax.set_title('Income by Occupation (Mean vs Median)', fontweight='bold')
ax.set_xlabel('Occupation')
ax.set_ylabel('Annual Income (USD)')
ax.set_xticks(x)
ax.set_xticklabels(occ_stats.index, rotation=15, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'${bar.get_height()/1000:.0f}K', ha='center', va='bottom', fontsize=7)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'${bar.get_height()/1000:.0f}K', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

## 6. Phân tích theo khu vực (Bar + Pie)

Thu nhập theo khu vực (trái) kết hợp với phân phối số lượng mẫu theo khu vực (phải).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Bar chart: Income by Location ──
loc_stats = df.groupby('Location')['Income'].agg(['mean', 'median']).sort_values('median', ascending=False)
x = np.arange(len(loc_stats))
width = 0.35
axes[0].bar(x - width/2, loc_stats['median'], width, label='Median', color=COLORS['info'], alpha=0.9)
axes[0].bar(x + width/2, loc_stats['mean'], width, label='Mean', color=COLORS['accent'], alpha=0.9)
axes[0].set_title('Income by Location Type', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(loc_stats.index)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[0].legend()

# ── Pie chart: Location Distribution ──
loc_counts = df['Location'].value_counts()
axes[1].pie(loc_counts, labels=loc_counts.index, autopct='%1.1f%%',
            colors=[COLORS['primary'], COLORS['secondary'], COLORS['accent']],
            startangle=90, textprops={'fontsize': 9})
axes[1].set_title('Location Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Phân tích tương quan (Correlation Heatmap)

Ma trận tương quan giữa các biến số — tương quan yếu với Income cho thấy mối quan hệ phi tuyến.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr_cols = ['Age', 'Number_of_Dependents', 'Work_Experience', 'Household_Size', 'Income']
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Correlation Coefficient'})
ax.set_title('Correlation Matrix — Numerical Features', fontweight='bold')

plt.tight_layout()
plt.show()

# Print correlation with Income
print('=== Correlation with Income ===')
income_corr = corr['Income'].drop('Income').sort_values(ascending=False)
for feat, c in income_corr.items():
    print(f'  {feat:30s} {c:+.4f}')

In [ ]:
# Correlation bar chart (with all encoded features)
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()
for col in cat_cols:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

all_corr = df_encoded.corr()['Income'].drop('Income').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors_corr = [COLORS['success'] if v > 0 else COLORS['danger'] for v in all_corr.values]
all_corr.plot(kind='barh', ax=ax, color=colors_corr, edgecolor='white', alpha=0.8)
ax.set_title('Correlation with Income (All Features)', fontweight='bold', fontsize=14)
ax.set_xlabel('Pearson Correlation Coefficient')
ax.axvline(0, color='gray', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

print('=== Full Correlation with Income ===')
for feat, c in all_corr.items():
    print(f'  {feat:40s} {c:+.4f}')

## 8. Phân tích tình trạng việc làm & giới tính

So sánh thu nhập Mean vs Median theo Employment Status và Gender.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
width = 0.35

# ── Employment Status ──
emp_stats = df.groupby('Employment_Status')['Income'].agg(['mean', 'median']).sort_values('mean', ascending=False)
x = np.arange(len(emp_stats))
axes[0].bar(x - width/2, emp_stats['median'], width, label='Median', color=COLORS['info'])
axes[0].bar(x + width/2, emp_stats['mean'], width, label='Mean', color=COLORS['danger'])
axes[0].set_title('Income by Employment Status', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(emp_stats.index, fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[0].legend()

# ── Gender ──
gender_stats = df.groupby('Gender')['Income'].agg(['mean', 'median'])
x = np.arange(len(gender_stats))
axes[1].bar(x - width/2, gender_stats['median'], width, label='Median', color=COLORS['secondary'])
axes[1].bar(x + width/2, gender_stats['mean'], width, label='Mean', color=COLORS['accent'])
axes[1].set_title('Income by Gender', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(gender_stats.index, fontsize=9)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Phân tích tình trạng hôn nhân & loại nhà ở

Thu nhập theo Marital Status và Housing Type — phát hiện xu hướng nhân khẩu học.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
width = 0.35

# ── Marital Status ──
mar_stats = df.groupby('Marital_Status')['Income'].agg(['mean', 'median']).sort_values('mean', ascending=False)
x = np.arange(len(mar_stats))
axes[0].bar(x - width/2, mar_stats['median'], width, label='Median', color=COLORS['primary'])
axes[0].bar(x + width/2, mar_stats['mean'], width, label='Mean', color=COLORS['accent'])
axes[0].set_title('Income by Marital Status', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(mar_stats.index, fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[0].legend()

# ── Housing Type ──
house_stats = df.groupby('Type_of_Housing')['Income'].agg(['mean', 'median']).sort_values('mean', ascending=False)
x = np.arange(len(house_stats))
axes[1].bar(x - width/2, house_stats['median'], width, label='Median', color=COLORS['success'])
axes[1].bar(x + width/2, house_stats['mean'], width, label='Mean', color=COLORS['danger'])
axes[1].set_title('Income by Housing Type', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(house_stats.index, fontsize=8, rotation=10, ha='right')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Phân tích tuổi & kinh nghiệm làm việc (Scatter Plot)

Biểu đồ phân tán với màu sắc theo **log(Income)** để phân biệt dải thu nhập rộng
($30K → $10M) — khám phá mối quan hệ giữa tuổi và kinh nghiệm.

> **Lưu ý:** Log scale ở đây chỉ dùng cho **màu sắc trực quan** trong EDA.
> Mô hình ML được huấn luyện trực tiếp trên **giá trị thu nhập gốc** (không log transform).


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(df['Work_Experience'], df['Age'], c=np.log1p(df['Income']),
                     cmap='viridis', alpha=0.4, s=8, edgecolors='none')
cbar = plt.colorbar(scatter, ax=ax, label='log1p(Income) — color scale only')
ax.set_xlabel('Work Experience (years)')
ax.set_ylabel('Age')
ax.set_title('Age vs Work Experience\n(color = log scale of income for visualization)', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Additional: Numerical features vs Income scatter plots
num_features = [c for c in num_cols if c != 'Income']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
scatter_colors = [COLORS['primary'], COLORS['secondary'], COLORS['success'], COLORS['accent']]

for i, col in enumerate(num_features):
    ax = axes[i // 2, i % 2]
    ax.scatter(df[col], df['Income'], alpha=0.15, s=8, color=scatter_colors[i])
    ax.set_title(f'{col} vs Income', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Income (USD)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

    # Add trend line
    z = np.polyfit(df[col], df['Income'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(x_line, p(x_line), 'r--', alpha=0.7, linewidth=2)

plt.tight_layout()
plt.suptitle('Numerical Features vs Income', y=1.02, fontsize=16, fontweight='bold')
plt.show()

## 11. Phân phối phần trăm thu nhập (Area Chart)

Đường cong percentile P5→P95 cho thấy sự chênh lệch thu nhập rất lớn ở các phần trăm cao.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
percentiles = list(range(5, 100, 5))
values = [df['Income'].quantile(p/100) for p in percentiles]

ax.fill_between(percentiles, values, alpha=0.3, color=COLORS['primary'])
ax.plot(percentiles, values, color=COLORS['primary'], linewidth=2, marker='o', markersize=5)

for i, (p, v) in enumerate(zip(percentiles, values)):
    if p in [25, 50, 75, 90, 95]:
        ax.annotate(f'P{p}: ${v:,.0f}', (p, v), textcoords='offset points',
                    xytext=(0, 12), ha='center', fontsize=7, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                              edgecolor=COLORS['primary'], alpha=0.8))

ax.set_xlabel('Percentile')
ax.set_ylabel('Income (USD)')
ax.set_title('Income Percentile Distribution', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Phân phối trình độ học vấn & nghề nghiệp (Doughnut Charts)

Biểu đồ donut thể hiện tỷ lệ phân phối các nhóm chính trong tập dữ liệu.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Education Level Doughnut ──
edu_counts = df['Education_Level'].value_counts()
wedges1, texts1, autotexts1 = axes[0].pie(
    edu_counts, labels=edu_counts.index, autopct='%1.1f%%',
    colors=COLORS['palette'][:len(edu_counts)], startangle=90,
    pctdistance=0.8, textprops={'fontsize': 9})
centre_circle1 = plt.Circle((0, 0), 0.55, fc='white')
axes[0].add_artist(centre_circle1)
axes[0].set_title('Education Level Distribution', fontweight='bold')
axes[0].text(0, 0, f'N={len(df):,}', ha='center', va='center', fontsize=10, fontweight='bold')

# ── Occupation Doughnut ──
occ_counts = df['Occupation'].value_counts()
wedges2, texts2, autotexts2 = axes[1].pie(
    occ_counts, labels=occ_counts.index, autopct='%1.1f%%',
    colors=COLORS['palette'][:len(occ_counts)], startangle=90,
    pctdistance=0.8, textprops={'fontsize': 9})
centre_circle2 = plt.Circle((0, 0), 0.55, fc='white')
axes[1].add_artist(centre_circle2)
axes[1].set_title('Occupation Distribution', fontweight='bold')
axes[1].text(0, 0, f'N={len(df):,}', ha='center', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Additional doughnut charts for other categorical features
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
other_cats = ['Employment_Status', 'Gender', 'Marital_Status',
              'Homeownership_Status', 'Type_of_Housing', 'Location']

for i, col in enumerate(other_cats):
    ax = axes[i // 3, i % 3]
    counts = df[col].value_counts()
    wedges, texts, autotexts = ax.pie(
        counts, labels=counts.index, autopct='%1.1f%%',
        colors=COLORS['palette'][:len(counts)], startangle=90,
        pctdistance=0.8, textprops={'fontsize': 8})
    centre = plt.Circle((0, 0), 0.55, fc='white')
    ax.add_artist(centre)
    ax.set_title(col.replace('_', ' '), fontweight='bold', fontsize=10)
    ax.text(0, 0, f'{len(counts)}\ncats', ha='center', va='center', fontsize=8)

plt.suptitle('Categorical Features Distribution', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

# PHẦN II — DATA PREPROCESSING & MODELING

---

---

# PHẦN II — DATA PREPROCESSING & MODELING

---


## 13. Data Preprocessing

Tiền xử lý dữ liệu gồm **3 bước chính** (không áp dụng log transform):

- **13.1** Label Encoding các biến phân loại → số nguyên
- **13.2** Feature Engineering (tạo 2 đặc trưng mới)
- **13.3** Tổng hợp feature matrix

> **Lưu ý:** Thu nhập có skewness ≈ 2.98 (lệch phải mạnh), nhưng các mô hình
> được huấn luyện **trực tiếp trên giá trị thu nhập gốc** (không log transform)
> để kết quả dự đoán dễ diễn giải hơn.


In [ ]:
# ── Income Statistics (No log transform) ──
df_clean = df.copy()

print('=== Income Statistics (Raw) ===')
print(f'  Mean:     ${df_clean["Income"].mean():>15,.2f}')
print(f'  Median:   ${df_clean["Income"].median():>15,.2f}')
print(f'  Std:      ${df_clean["Income"].std():>15,.2f}')
print(f'  Skewness: {df_clean["Income"].skew():>15.4f}')
print(f'  Min:      ${df_clean["Income"].min():>15,.0f}')
print(f'  Max:      ${df_clean["Income"].max():>15,.0f}')
print()
print('  → Models trained directly on raw income (no log transform).')


### 13.1 Label Encoding biến phân loại

Chuyển đổi 9 biến phân loại (text) thành dạng số nguyên.
Phù hợp với Random Forest vì cây quyết định không cần thứ tự tuyến tính.


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ── Label Encoding ──
label_encoders = {}
encoding_maps = {}

for col in cat_cols:
    le = LabelEncoder()
    df_clean[col + '_encoded'] = le.fit_transform(df_clean[col])
    label_encoders[col] = le
    encoding_maps[col] = dict(zip(le.classes_.tolist(), le.transform(le.classes_).tolist()))

print('=== Label Encoding Results ===')
for col in cat_cols:
    print(f'  {col}:')
    print(f'    {encoding_maps[col]}')

### 13.2 Feature Engineering

Tạo 2 đặc trưng mới nhằm nắm bắt thêm thông tin từ dữ liệu gốc:
- `Age_WorkExp_Ratio`: phản ánh thời điểm bắt đầu sự nghiệp (tuổi / kinh nghiệm)
- `Dependents_per_HHSize`: tỷ lệ người phụ thuộc trong hộ gia đình


In [ ]:
# ── Feature Engineering ──
df_clean['Age_WorkExp_Ratio'] = df_clean['Age'] / (df_clean['Work_Experience'] + 1)
df_clean['Dependents_per_HHSize'] = df_clean['Number_of_Dependents'] / df_clean['Household_Size']

print('=== Engineered Features ===')
print(f"  Age_WorkExp_Ratio      — mean: {df_clean['Age_WorkExp_Ratio'].mean():.3f}  std: {df_clean['Age_WorkExp_Ratio'].std():.3f}")
print(f"  Dependents_per_HHSize  — mean: {df_clean['Dependents_per_HHSize'].mean():.3f}  std: {df_clean['Dependents_per_HHSize'].std():.3f}")


In [ ]:
# ── Build final feature list ──
feature_cols = [c for c in num_cols if c != 'Income']
for col in cat_cols:
    feature_cols.append(col + '_encoded')
feature_cols.extend(['Age_WorkExp_Ratio', 'Dependents_per_HHSize'])

X = df_clean[feature_cols]
y = df_clean['Income']   # raw income — no log transform

print(f'Feature columns ({len(feature_cols)}): {feature_cols}')
print(f'Target: raw Income  (mean=${y.mean():,.0f}  median=${y.median():,.0f})')


## 14. Train / Validation / Test Split (70 / 15 / 15)

Chia dữ liệu thành **3 tập** với `random_state=42`:

| Tập | Tỷ lệ | Số mẫu | Mục đích |
|-----|-------|--------|----------|
| Train | 70% | 7,000 | Huấn luyện mô hình |
| Validation | 15% | 1,500 | Chọn mô hình tốt nhất |
| Test | 15% | 1,500 | Đánh giá cuối cùng |

**Lưu ý:** Model selection dựa trên **Validation R²** (không phải Test).
Test set chỉ được dùng một lần duy nhất để báo cáo kết quả cuối.


In [ ]:
from sklearn.model_selection import train_test_split

# ── First split: 70% train, 30% temp ──
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)

# ── Second split: 15% val + 15% test (50/50 of temp) ──
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

total = len(X)
print('=== Data Split (70 / 15 / 15) ===')
print(f'  Train:      {len(X_train):>5,} samples ({len(X_train)/total*100:.1f}%)')
print(f'  Validation: {len(X_val):>5,} samples ({len(X_val)/total*100:.1f}%)')
print(f'  Test:       {len(X_test):>5,} samples ({len(X_test)/total*100:.1f}%)')
print()
print(f'  Train  y — mean: ${y_train.mean():,.0f}  median: ${y_train.median():,.0f}')
print(f'  Val    y — mean: ${y_val.mean():,.0f}  median: ${y_val.median():,.0f}')
print(f'  Test   y — mean: ${y_test.mean():,.0f}  median: ${y_test.median():,.0f}')


## 15. Feature Scaling

Chuẩn hóa đặc trưng bằng `StandardScaler` (mean=0, std=1).

**Lưu ý:**
- Scaler **chỉ fit trên Train set** để tránh data leakage
- Val và Test set chỉ được `transform` (không `fit`)
- Random Forest và Gradient Boosting **không yêu cầu scaling**
  (chỉ dùng cho Linear, Ridge, Lasso)


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print('=== Feature Scaling (StandardScaler) ===')
print(f'  Train scaled — mean: {X_train_scaled.mean():.4f}  std: {X_train_scaled.std():.4f}')
print(f'  Val   scaled — mean: {X_val_scaled.mean():.4f}  std: {X_val_scaled.std():.4f}')
print(f'  Test  scaled — mean: {X_test_scaled.mean():.4f}  std: {X_test_scaled.std():.4f}')


## 16. Model Training

Huấn luyện **5 mô hình hồi quy** trực tiếp trên **giá trị thu nhập gốc** (không log transform):

1. **Linear Regression** — Baseline tuyến tính
2. **Ridge Regression** (α=10) — L2 regularization
3. **Lasso Regression** (α=1) — L1 regularization + feature selection
4. **Random Forest** (300 trees) — Ensemble cây quyết định
5. **Gradient Boosting** (300 trees) — Boosting sequential

**Chiến lược đánh giá:**
- Mô hình tốt nhất được chọn dựa trên **Validation R²**
- Kết quả Test set chỉ dùng để báo cáo cuối cùng


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Define models ──
models_config = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=10.0),
    'Lasso Regression':  Lasso(alpha=0.01),
    'Random Forest':     RandomForestRegressor(
                             n_estimators=300, max_depth=20,
                             min_samples_split=5, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(
                             n_estimators=300, max_depth=6,
                             learning_rate=0.05, subsample=0.8, random_state=42),
}

print(f'Models to train: {len(models_config)}')
for name, model in models_config.items():
    print(f'  • {name}: {type(model).__name__}')

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def calc_metrics(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2   = r2_score(y_true, y_pred)
    nz   = y_true != 0
    mape = float(np.mean(np.abs((y_true[nz] - y_pred[nz]) / y_true[nz])) * 100)
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape}

# ── Train all models & evaluate on Validation set ──
results = {}
trained_models = {}
best_val_r2 = -np.inf
best_name = None

for name, model in models_config.items():
    is_tree = 'Forest' in name or 'Boosting' in name

    if is_tree:
        model.fit(X_train, y_train)
        y_val_pred  = model.predict(X_val)
        y_test_pred = model.predict(X_test)
        cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2', n_jobs=-1)
    else:
        model.fit(X_train_scaled, y_train)
        y_val_pred  = model.predict(X_val_scaled)
        y_test_pred = model.predict(X_test_scaled)
        cv_scores = cross_val_score(model, scaler.transform(X), y, cv=5, scoring='r2')

    val_m  = calc_metrics(y_val.values,  y_val_pred)
    test_m = calc_metrics(y_test.values, y_test_pred)

    results[name] = {
        'Val_R2':   val_m['R2'],   'Val_MAE':  val_m['MAE'],
        'Val_RMSE': val_m['RMSE'], 'Val_MAPE': val_m['MAPE'],
        'Test_R2':  test_m['R2'],  'Test_MAE': test_m['MAE'],
        'Test_RMSE':test_m['RMSE'],'Test_MAPE':test_m['MAPE'],
        'CV_R2_Mean': cv_scores.mean(), 'CV_R2_Std': cv_scores.std(),
        'y_val_pred':  y_val_pred,
        'y_test_pred': y_test_pred,
    }
    trained_models[name] = model

    if val_m['R2'] > best_val_r2:
        best_val_r2 = val_m['R2']
        best_name   = name

    print(f"  {name}:")
    print(f"    Val  R2={val_m['R2']:.4f}  MAE=${val_m['MAE']:,.0f}  RMSE=${val_m['RMSE']:,.0f}  MAPE={val_m['MAPE']:.1f}%")
    print(f"    Test R2={test_m['R2']:.4f}  MAE=${test_m['MAE']:,.0f}  RMSE=${test_m['RMSE']:,.0f}")
    print(f"    CV   R2={cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print()

best_model   = trained_models[best_name]
best_results = results[best_name]
print(f"★ Best Model (by Validation R²): {best_name}")
print(f"  Val  R2 = {best_results['Val_R2']:.4f}  MAE=${best_results['Val_MAE']:,.0f}")
print(f"  Test R2 = {best_results['Test_R2']:.4f}  MAE=${best_results['Test_MAE']:,.0f}")
print(f"  CV   R2 = {best_results['CV_R2_Mean']:.4f} ± {best_results['CV_R2_Std']:.4f}")


## 17. Model Evaluation

Đánh giá chi tiết hiệu suất mô hình thông qua:
- **17.1** Bảng so sánh tổng hợp (Val + Test)
- **17.2** Biểu đồ so sánh R² và Cross-Validation R²
- **17.3** Biểu đồ so sánh MAE & RMSE
- **17.4** Actual vs Predicted & Residual Distribution
- **17.5** Feature Importance


### 17.1 Bảng so sánh tổng hợp

In [ ]:
import pandas as pd

# ── Comparison table ──
comparison_data = []
for name in results:
    r = results[name]
    best_marker = ' ★' if name == best_name else ''
    comparison_data.append({
        'Model':      name + best_marker,
        'Val R²':     round(r['Val_R2'],   4),
        'Val MAE':    f"${r['Val_MAE']:,.0f}",
        'Val RMSE':   f"${r['Val_RMSE']:,.0f}",
        'Test R²':    round(r['Test_R2'],  4),
        'Test MAE':   f"${r['Test_MAE']:,.0f}",
        'CV R²':      f"{r['CV_R2_Mean']:.4f} ± {r['CV_R2_Std']:.4f}",
    })

comp_df = pd.DataFrame(comparison_data).set_index('Model')
print(comp_df.to_string())


### 17.2 So sánh Val R² và Cross-Validation R²


In [ ]:
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

model_names  = list(results.keys())
colors_bar   = [COLORS['accent'] if n == best_name else COLORS['primary'] for n in model_names]
val_r2_vals  = [results[m]['Val_R2']      for m in model_names]
cv_r2_vals   = [results[m]['CV_R2_Mean']  for m in model_names]
cv_r2_stds   = [results[m]['CV_R2_Std']   for m in model_names]

# ── Val R² ──
bars1 = axes[0].bar(range(len(model_names)), val_r2_vals, color=colors_bar, alpha=0.9, edgecolor='white')
axes[0].set_xticks(range(len(model_names)))
axes[0].set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=8)
axes[0].set_ylabel('R² Score')
axes[0].set_title('Model Comparison — Validation R²', fontweight='bold')
axes[0].axhline(0, color='gray', linewidth=0.8, linestyle='--')
for bar, val in zip(bars1, val_r2_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=8)

# ── CV R² ──
bars2 = axes[1].bar(range(len(model_names)), cv_r2_vals, yerr=cv_r2_stds,
                     color=colors_bar, alpha=0.9, edgecolor='white',
                     capsize=5, error_kw={'linewidth': 1.5})
axes[1].set_xticks(range(len(model_names)))
axes[1].set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=8)
axes[1].set_ylabel('CV R² Score')
axes[1].set_title('Model Comparison — CV R² (5-fold)', fontweight='bold')
axes[1].axhline(0, color='gray', linewidth=0.8, linestyle='--')
for bar, val in zip(bars2, cv_r2_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()


### 17.3 So sánh MAE & RMSE

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

mae_vals  = [results[m]['Val_MAE']  for m in model_names]
rmse_vals = [results[m]['Val_RMSE'] for m in model_names]

# ── MAE ──
bars_mae = axes[0].bar(range(len(model_names)), mae_vals, color=colors_bar, alpha=0.9, edgecolor='white')
axes[0].set_xticks(range(len(model_names)))
axes[0].set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=8)
axes[0].set_ylabel('MAE (USD)')
axes[0].set_title('Model Comparison — Validation MAE', fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.2f}M'))
for bar, val in zip(bars_mae, mae_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                 f'${val/1000:,.0f}K', ha='center', va='bottom', fontsize=7)

# ── RMSE ──
bars_rmse = axes[1].bar(range(len(model_names)), rmse_vals, color=colors_bar, alpha=0.9, edgecolor='white')
axes[1].set_xticks(range(len(model_names)))
axes[1].set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=8)
axes[1].set_ylabel('RMSE (USD)')
axes[1].set_title('Model Comparison — Validation RMSE', fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.2f}M'))
for bar, val in zip(bars_rmse, rmse_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                 f'${val/1000:,.0f}K', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()


### 17.4 Actual vs Predicted & Residual Distribution (Test Set)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

y_test_pred = best_results['y_test_pred']

# ── Actual vs Predicted (raw income) ──
axes[0].scatter(y_test, y_test_pred, alpha=0.3, s=8, color=COLORS['primary'])
min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Perfect Prediction')
axes[0].set_xlabel('Actual Income (USD)')
axes[0].set_ylabel('Predicted Income (USD)')
axes[0].set_title(
    f'Actual vs Predicted — {best_name}\n'
    f'Test R² = {best_results["Test_R2"]:.4f}  MAE=${best_results["Test_MAE"]:,.0f}',
    fontweight='bold'
)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[0].legend()

# ── Residual Distribution ──
residuals = y_test.values - y_test_pred
axes[1].hist(residuals, bins=40, color=COLORS['secondary'], alpha=0.85, edgecolor='white')
axes[1].axvline(0, color=COLORS['danger'], linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual (Actual - Predicted, USD)')
axes[1].set_ylabel('Frequency')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[1].set_title(
    f'Residual Distribution\nMean=${residuals.mean():,.0f}  Std=${residuals.std():,.0f}',
    fontweight='bold'
)

plt.tight_layout()
plt.show()


### 17.5 Feature Importance


In [ ]:
import numpy as np

# ── Feature Importance: tree-based models use feature_importances_,
#    linear models use normalized absolute coefficients ──
if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    fi_source = f'feature_importances_ ({best_name})'
else:
    # Fallback: use Ridge coefficients
    ridge_model = trained_models.get('Ridge Regression')
    raw = pd.Series(np.abs(ridge_model.coef_), index=feature_cols)
    fi = (raw / raw.sum()).sort_values(ascending=False)
    fi_source = 'normalized |coef| (Ridge Regression)'

print(f'=== Feature Importance — {fi_source} ===')
for rank, (feat, imp) in enumerate(fi.items(), 1):
    feat_clean = feat.replace('_encoded', '').replace('_', ' ')
    bar = '█' * int(imp * 100)
    print(f'  {rank:2d}. {feat_clean:35s} {imp:.4f} ({imp*100:.2f}%)  {bar}')


In [ ]:
# ── Feature Importance Chart (Horizontal Bar) ──
fig, ax = plt.subplots(figsize=(10, 6))
fi_sorted = fi.sort_values(ascending=True)
colors_fi = plt.cm.viridis(np.linspace(0.2, 0.9, len(fi_sorted)))

bars = ax.barh(range(len(fi_sorted)), fi_sorted.values, color=colors_fi,
               edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(fi_sorted)))
feature_labels = [f.replace('_encoded', '').replace('_', ' ') for f in fi_sorted.index]
ax.set_yticklabels(feature_labels, fontsize=9)
ax.set_xlabel('Importance Score')
ax.set_title(f'Feature Importance — {fi_source}', fontweight='bold')

for i, (val, bar) in enumerate(zip(fi_sorted.values, bars)):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.2%}',
            va='center', fontsize=8, fontweight='bold')

ax.set_xlim(0, fi_sorted.max() * 1.25)
plt.tight_layout()
plt.show()


### Ghi chú về hiệu suất mô hình

> **Tại sao R² thấp?**
>
> Dataset này có thu nhập lệch phải rất mạnh (skewness ≈ 2.98, mean $816K vs median $73K).
> Khi không áp dụng log transform, các mô hình bị ảnh hưởng nặng bởi outliers (income > $5M).
> Đây là trade-off có chủ đích: kết quả dự đoán dễ diễn giải hơn (đơn vị USD gốc),
> nhưng R² thấp hơn so với khi train trên log(Income).
>
> Với dataset synthetic này, R² ≈ 0.01 là dự kiến và nhất quán qua các fold CV.


In [ ]:
# ── Val vs Test Metrics Comparison ──
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_to_plot = [
    ('R2',   'R² Score',      False),
    ('MAE',  'MAE (USD)',     True),
    ('RMSE', 'RMSE (USD)',    True),
]

for ax, (metric, label, use_dollar) in zip(axes, metrics_to_plot):
    val_vals  = [results[m][f'Val_{metric}']  for m in model_names]
    test_vals = [results[m][f'Test_{metric}'] for m in model_names]
    x = np.arange(len(model_names))
    w = 0.35

    bars_val  = ax.bar(x - w/2, val_vals,  w, label='Validation', color=COLORS['primary'],  alpha=0.9, edgecolor='white')
    bars_test = ax.bar(x + w/2, test_vals, w, label='Test',        color=COLORS['secondary'], alpha=0.9, edgecolor='white')

    ax.set_xticks(x)
    ax.set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=7)
    ax.set_ylabel(label)
    ax.set_title(f'{label} — Val vs Test', fontweight='bold')
    ax.legend(fontsize=8)
    ax.axhline(0, color='gray', linewidth=0.6, linestyle='--')

    if use_dollar:
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

plt.tight_layout()
plt.show()


---

## 18. Tổng kết & Khuyến nghị

### Dataset Characteristics
- **10,000 records** với **13 features** (4 numerical + 9 categorical) và 1 target (Income)
- **Không có missing values** và **không có duplicate** rows
- Dataset **synthetic** với nhiễu ngẫu nhiên cao

### Income Distribution
- Thu nhập lệch phải rất mạnh (skewness ≈ 2.98)
- Mean ($816K) cao hơn Median ($73K) khoảng **11 lần** → outliers cực đoan
- ~19% dữ liệu là outliers (theo IQR method)
- **Không áp dụng log transform** — mô hình train trực tiếp trên raw income

### Data Split: 70 / 15 / 15
- Train: 7,000 mẫu — Validation: 1,500 mẫu — Test: 1,500 mẫu
- Model selection dựa trên **Validation R²**
- Test set chỉ dùng để báo cáo kết quả cuối cùng

### Feature Relationships
- **Tương quan tuyến tính yếu** giữa features và income (max |r| < 0.2)
- Engineered feature **Age/WorkExp Ratio** là predictor quan trọng nhất

### Model Results (Val Set — Raw Income)
| Model | Val R² | Val MAE | Test R² | CV R² |
|-------|--------|---------|---------|-------|
| **Lasso Regression ★** | **0.0114** | $1,093,787 | 0.0077 | 0.0094 |
| Linear Regression | 0.0114 | $1,093,787 | 0.0077 | 0.0094 |
| Ridge Regression | 0.0114 | $1,093,823 | 0.0077 | 0.0094 |
| Random Forest | -0.0043 | $1,151,547 | 0.0109 | 0.0142 |
| Gradient Boosting | -0.0487 | $1,131,649 | -0.0429 | -0.0358 |

> **Lưu ý:** R² thấp là dự kiến khi train trên raw income với dataset synthetic
> lệch phải mạnh. Mô hình tuyến tính (Lasso) outperform tree-based trên validation,
> phản ánh tương quan tuyến tính yếu và nhiễu cao trong dữ liệu.

### Top Feature Importance (Ridge |coef| normalized)
1. **Age/WorkExp Ratio** (17.78%) — thời điểm bắt đầu sự nghiệp
2. **Location** (17.03%) — khu vực địa lý
3. **Homeownership Status** (16.16%) — tình trạng sở hữu nhà
4. **Work Experience** (15.14%) — số năm kinh nghiệm

### Khuyến nghị cải thiện
1. **Thử Log Transform** — nếu ưu tiên R² cao hơn tính diễn giải
2. **Thu thập thêm dữ liệu** — bổ sung biến tài sản, khoản vay, lịch sử thu nhập
3. **Hyperparameter Tuning** — RandomizedSearchCV hoặc Bayesian Optimization
4. **Stacking/Blending** — kết hợp nhiều mô hình
5. **Xử lý Outlier nâng cao** — Winsorization hoặc Robust Scaler
6. **Feature Engineering** — thêm interaction features


In [ ]:
print('=' * 60)
print('  📊 EDA & Modeling Complete!')
print('  Charts are consistent with Word report')
print('  (Bao_cao_Phan_tich_Thu_nhap_Ho_Gia_dinh.docx)')
print()
print(f'  🏆 Best Model (Val R²): {best_name}')
print(f'     Val  R²:      {best_results["Val_R2"]:.4f}')
print(f'     Test R²:      {best_results["Test_R2"]:.4f}')
print(f'     CV R² (5-fold): {best_results["CV_R2_Mean"]:.4f} ± {best_results["CV_R2_Std"]:.4f}')
print(f'     Val  MAE:     ${best_results["Val_MAE"]:,.0f}')
print(f'     Test MAE:     ${best_results["Test_MAE"]:,.0f}')
print()
print('  Split: 70% Train / 15% Val / 15% Test')
print('  Target: Raw Income (no log transform)')
print()
print('  To run the web app:')
print('    python main.py')
print('=' * 60)
